# Treja — Fine-tune RTMDet-Small on 234-class taxonomy

**Run this whole notebook top to bottom: `Runtime` → `Run all`.**

What it does, in order:
1. Downloads a balanced set of training images for all 234 target classes from Open Images V7 (free, CC-licensed).
2. Converts them to the format the trainer needs.
3. Fine-tunes RTMDet-Small starting from your existing COCO checkpoint (not from scratch).
4. Saves a checkpoint every epoch, so nothing is lost if the session drops.
5. At the end: exports to the same raw-ONNX format as before, verifies it, and **pushes the result straight to your GitHub repo** — you don't need to do anything after clicking Run all besides keeping the tab open.

**Before running:** make sure you've added `GITHUB_TOKEN` in Colab's Secrets panel (key icon, left sidebar) with write access to `treja-poc`, and toggled Notebook access on for this notebook.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118

# fiftyone installed on its own, isolated from the mmdet stack's install below —
# keeps its error (if any) clean and separate rather than buried in a combined resolve.
!pip install fiftyone 2>&1 | tail -30

!pip install -q -U openmim
!mim install -q mmengine
!mim install -q "mmcv==2.1.0"
!pip install -q mmdet==3.3.0
!pip install -q onnx onnxruntime

# numpy/opencv pinned LAST — mmcv/torch need numpy<2, and installing this after
# everything else stops a later package from silently upgrading numpy and breaking mmcv.
# (This exact fix already worked once before, for the same conflict, in the first export notebook.)
!pip uninstall -y -q opencv-python opencv-python-headless 2>/dev/null
!pip install -q "numpy<2" "opencv-python-headless==4.9.0.80"


## 1b. Verify every dependency imports cleanly before continuing

In [ ]:
# Fail fast and clearly: check every critical import right now, before the slow
# data-download step, so a broken install is obvious immediately, not 20 minutes in.
import importlib
print('=== Dependency check ===')
for pkg in ['numpy', 'torch', 'mmengine', 'mmcv', 'mmdet', 'fiftyone', 'onnx', 'onnxruntime', 'cv2']:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        print(f'OK   {pkg:15s} {v}')
    except Exception as e:
        print(f'FAIL {pkg:15s} {type(e).__name__}: {e}')
print('=== If any FAIL above, stop here and send that line back before continuing. ===')

## 2. Load the 234-class taxonomy directly from GitHub (single source of truth)

In [ ]:
import urllib.request
url = 'https://raw.githubusercontent.com/Noamgordon/treja-poc/main/model-export/target_taxonomy_final.txt'
CLASSES = [l.strip() for l in urllib.request.urlopen(url).read().decode().splitlines() if l.strip()]
print(f'{len(CLASSES)} classes loaded.')
assert len(CLASSES) == 234, f'Expected 234, got {len(CLASSES)} — taxonomy file may have changed, stop and check.'

## 3. Download a balanced training set from Open Images V7

`max_samples` in FiftyOne's Open Images loader is a **global** cap, not per-class — calling it once with all 234 classes would let common classes (like Chair) drown out rare ones. So we download **one class at a time** and merge, to keep classes roughly balanced. 120 images/class is a size/time compromise for a free-tier run — expect this step to take a while, it's downloading real images.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

IMAGES_PER_CLASS = 120

merged = fo.Dataset('treja_train', overwrite=True)
failed_classes = []
for i, cls in enumerate(CLASSES):
    try:
        part = foz.load_zoo_dataset(
            'open-images-v7',
            split='train',
            label_types=['detections'],
            classes=[cls],
            max_samples=IMAGES_PER_CLASS,
            dataset_name=f'oiv7_{i}',
        )
        merged.merge_samples(part)
        fo.delete_dataset(part.name)
        if i % 10 == 0:
            print(f'[{i+1}/{len(CLASSES)}] done. Merged dataset now has {len(merged)} images.')
    except Exception as e:
        print(f'FAILED on class "{cls}": {e}')
        failed_classes.append(cls)

print(f'Done. {len(merged)} total images. {len(failed_classes)} classes failed: {failed_classes}')

## 4. Restrict labels to our 234 classes only and export to COCO format

Open Images images often contain other objects too (people, animals, etc.) that we deliberately excluded — filter those labels out before exporting, so the model is never trained to associate those pixels with a class we don't want.

In [ ]:
from fiftyone import ViewField as F

filtered = merged.filter_labels('ground_truth', F('label').is_in(CLASSES))

import os, shutil
export_dir = '/content/coco_export'
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)

filtered.export(
    export_dir=export_dir,
    dataset_type=fo.types.COCODetectionDataset,
    label_field='ground_truth',
    classes=CLASSES,
)
print('Exported to', export_dir)
!ls {export_dir}

## 5. Download the existing COCO-pretrained checkpoint and configs (same as the first export notebook)

In [ ]:
!mkdir -p /content/ckpt
!wget -q "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_s_8xb32-300e_coco/rtmdet_s_8xb32-300e_coco_20220905_161602-387a891e.pth" -O /content/ckpt/rtmdet_s.pth
!git clone --depth 1 --filter=blob:none --sparse https://github.com/open-mmlab/mmdetection.git /content/mmdetection-configs
%cd /content/mmdetection-configs
!git sparse-checkout set configs
%cd /content
print('Ready:', os.path.exists('/content/ckpt/rtmdet_s.pth'))

## 6. Build the fine-tuning config

Starts from the same checkpoint the deployed app already uses, just changes the number of output classes (80 → 234) and points training at our new dataset. Fewer epochs than a from-scratch run (this is fine-tuning, not training from zero) and a checkpoint saved every epoch.

In [ ]:
from mmengine import Config

cfg = Config.fromfile('/content/mmdetection-configs/configs/rtmdet/rtmdet_s_8xb32-300e_coco.py')

NUM_CLASSES = len(CLASSES)
cfg.model.bbox_head.num_classes = NUM_CLASSES
cfg.load_from = '/content/ckpt/rtmdet_s.pth'

metainfo = dict(classes=tuple(CLASSES))
for split in ['train_dataloader', 'val_dataloader', 'test_dataloader']:
    if split in cfg:
        cfg[split].dataset.metainfo = metainfo

cfg.train_dataloader.dataset.ann_file = f'{export_dir}/labels.json'
cfg.train_dataloader.dataset.data_prefix = dict(img=f'{export_dir}/data/')
cfg.train_dataloader.dataset.data_root = ''
cfg.train_dataloader.batch_size = 8
cfg.train_dataloader.num_workers = 2

# We only have a train split in this quick fine-tune pass — skip val/test loops entirely.
cfg.val_dataloader = None
cfg.val_cfg = None
cfg.val_evaluator = None
cfg.test_dataloader = None
cfg.test_cfg = None
cfg.test_evaluator = None

EPOCHS = 20
cfg.train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=EPOCHS, val_interval=EPOCHS + 1)
cfg.param_scheduler = [
    dict(type='LinearLR', start_factor=1e-5, by_epoch=False, begin=0, end=500),
    dict(type='CosineAnnealingLR', eta_min=0.0002 * 0.05, begin=EPOCHS // 2, end=EPOCHS,
         T_max=EPOCHS // 2, by_epoch=True, convert_to_iter_based=True),
]
cfg.optim_wrapper.optimizer.lr = 0.0004  # lower than from-scratch LR, since we're fine-tuning

cfg.default_hooks.checkpoint = dict(type='CheckpointHook', interval=1, max_keep_ckpts=2, save_best=None)
cfg.default_hooks.logger = dict(type='LoggerHook', interval=20)
cfg.work_dir = '/content/work_dir'
cfg.train_dataloader.dataset.filter_cfg = dict(filter_empty_gt=True, min_size=0)

print(f'Fine-tuning for {EPOCHS} epochs, {NUM_CLASSES} classes, checkpoint saved every epoch to {cfg.work_dir}')

## 7. Auto-push checkpoints to GitHub as training progresses

A custom hook that commits+pushes the latest checkpoint after every epoch, using the `GITHUB_TOKEN` secret. This means even a mid-training disconnect doesn't lose real progress — the last completed epoch is already safely in your repo.

In [ ]:
from google.colab import userdata
import subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/Noamgordon/treja-poc.git'

if not os.path.exists('/content/repo'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/repo'], check=True)
subprocess.run(['git', 'config', 'user.email', 'colab@treja.local'], cwd='/content/repo')
subprocess.run(['git', 'config', 'user.name', 'Treja Colab Bot'], cwd='/content/repo')

from mmengine.hooks import Hook
from mmengine.registry import HOOKS

@HOOKS.register_module()
class GitPushHook(Hook):
    def after_train_epoch(self, runner):
        try:
            ckpt_dir = '/content/repo/model-export/finetune_checkpoints'
            os.makedirs(ckpt_dir, exist_ok=True)
            latest = os.path.join(runner.work_dir, 'epoch_' + str(runner.epoch) + '.pth')
            if os.path.exists(latest):
                shutil.copy(latest, os.path.join(ckpt_dir, 'latest_checkpoint.pth'))
                with open(os.path.join(ckpt_dir, 'progress.txt'), 'w') as f:
                    f.write(f'epoch {runner.epoch} / {EPOCHS} complete\n')
                subprocess.run(['git', 'add', '-A'], cwd='/content/repo', check=True)
                r = subprocess.run(['git', 'commit', '-m', f'Training checkpoint: epoch {runner.epoch}/{EPOCHS}'],
                                    cwd='/content/repo')
                subprocess.run(['git', 'push'], cwd='/content/repo', check=True)
                print(f'Pushed checkpoint for epoch {runner.epoch}')
        except Exception as e:
            print(f'GitPushHook failed (training continues regardless): {e}')

cfg.custom_hooks = cfg.get('custom_hooks', []) + [dict(type='GitPushHook')]
print('Auto-push configured.')

## 8. Train

In [ ]:
from mmengine.runner import Runner

runner = Runner.from_cfg(cfg)
runner.train()
print('Training complete.')

## 9. Export the fine-tuned model to ONNX (same verified method as the first notebook) and push

In [ ]:
import torch
from mmdet.apis import init_detector
from mmdet.structures.bbox import distance2bbox

final_ckpt = sorted(
    [f for f in os.listdir(cfg.work_dir) if f.startswith('epoch_') and f.endswith('.pth')],
    key=lambda x: int(x.split('_')[1].split('.')[0]),
)[-1]
final_ckpt_path = os.path.join(cfg.work_dir, final_ckpt)
print('Using final checkpoint:', final_ckpt_path)

cfg.dump('/content/finetuned_config.py')
model = init_detector('/content/finetuned_config.py', final_ckpt_path, device='cpu')
model.eval()

class RTMDetRawExport(torch.nn.Module):
    def __init__(self, det_model):
        super().__init__()
        self.backbone = det_model.backbone
        self.neck = det_model.neck
        self.bbox_head = det_model.bbox_head

    def forward(self, img):
        feats = self.neck(self.backbone(img))
        cls_scores, bbox_preds = self.bbox_head(feats)
        featmap_sizes = [c.shape[-2:] for c in cls_scores]
        mlvl_priors = self.bbox_head.prior_generator.grid_priors(
            featmap_sizes, dtype=cls_scores[0].dtype, device=cls_scores[0].device)
        flatten_cls, flatten_bbox = [], []
        for cls_score, bbox_pred in zip(cls_scores, bbox_preds):
            b, c, h, w = cls_score.shape
            flatten_cls.append(cls_score.permute(0, 2, 3, 1).reshape(b, h * w, c))
            bb, bc, bh, bw = bbox_pred.shape
            flatten_bbox.append(bbox_pred.permute(0, 2, 3, 1).reshape(bb, bh * bw, bc))
        cls_scores_cat = torch.cat(flatten_cls, dim=1)
        bbox_preds_cat = torch.cat(flatten_bbox, dim=1)
        priors_cat = torch.cat(mlvl_priors, dim=0)
        bboxes = distance2bbox(priors_cat[None, :, :2], bbox_preds_cat)
        scores = cls_scores_cat.sigmoid()
        return bboxes, scores

export_model = RTMDetRawExport(model)
export_model.eval()
dummy = torch.randn(1, 3, 640, 640)
with torch.no_grad():
    b, s = export_model(dummy)
print('Output shapes -> boxes:', b.shape, 'scores:', s.shape, f'(expect [.., {NUM_CLASSES}] on scores)')
assert s.shape[-1] == NUM_CLASSES, 'Class count mismatch — stop and investigate before pushing.'

torch.onnx.export(
    export_model, dummy, '/content/model_234class.onnx',
    input_names=['image'], output_names=['boxes', 'scores'],
    opset_version=17, do_constant_folding=True,
)
import onnx
onnx_model = onnx.load('/content/model_234class.onnx')
onnx.checker.check_model(onnx_model)
print('ONNX export valid. Size:', os.path.getsize('/content/model_234class.onnx') / 1e6, 'MB')

In [ ]:
import json

shutil.copy('/content/model_234class.onnx', '/content/repo/docs/model_234class.onnx')
js_content = 'const COCO_CLASSES = ' + json.dumps(CLASSES) + ';\n'
with open('/content/repo/docs/classes_234.js', 'w') as f:
    f.write(js_content)

with open('/content/repo/model-export/finetune_checkpoints/progress.txt', 'w') as f:
    f.write('TRAINING COMPLETE — model_234class.onnx and classes_234.js pushed to docs/\n')

subprocess.run(['git', 'add', '-A'], cwd='/content/repo', check=True)
subprocess.run(['git', 'commit', '-m', 'Fine-tuned 234-class model: final ONNX export + class list'],
                cwd='/content/repo', check=True)
subprocess.run(['git', 'push'], cwd='/content/repo', check=True)
print('DONE. Final model pushed to docs/model_234class.onnx on GitHub.')